Exercise 1 : Multi-Layer Perceptron (MLP) Classifier

Aim: Develop a MLP model usign PyTorch to classify CIFAR-10 images

Dataset: CIFAR-10  contains 60,000 colour images across 10 classes, including 50,000 training images and 10,000 test images.

Approach: Train the MLP using the training data and use validation data to monitor its performance.

Evaluation: Assess the model usign accuracy and loss values, supported by training and validation loss graphs.

Dataset Loading

In [2]:
import sys

print("Python version:", sys.version)

try:
    import torch
    import torchvision

    print("PyTorch:", torch.__version__)
    print("TorchVision:", torchvision.__version__)

except ImportError as e:
    print("Package error:", e)

Python version: 3.14.0 (tags/v3.14.0:ebf955d, Oct  7 2025, 10:15:03) [MSC v.1944 64 bit (AMD64)]
PyTorch: 2.15.0+cpu
TorchVision: 0.29.2+cpu


In [ ]:
import torch
import torchvision

print("PyTorch:", torch.__version__)
print("TorchVision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch: 2.15.0+cpu
TorchVision: 0.29.2+cpu
CUDA available: False


In [3]:
import ssl
import sys
import torchvision
from torchvision import datasets

print("Python version:", sys.version)
print("TorchVision version:", torchvision.__version__)
print("CIFAR-10 download URL:", datasets.CIFAR10.url)
print("SSL verification:", ssl.create_default_context().verify_mode)
print("Certificate paths:", ssl.get_default_verify_paths())

try:
    import certifi
    print("Certifi version:", certifi.__version__)
    print("Certifi certificate file:", certifi.where())
except ImportError:
    print("Certifi is not installed.")

Python version: 3.14.0 (tags/v3.14.0:ebf955d, Oct  7 2025, 10:15:03) [MSC v.1944 64 bit (AMD64)]
TorchVision version: 0.29.2+cpu
CIFAR-10 download URL: https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz
SSL verification: 2
Certificate paths: DefaultVerifyPaths(cafile=None, capath=None, openssl_cafile_env='SSL_CERT_FILE', openssl_cafile='C:\\Program Files\\Common Files\\SSL/cert.pem', openssl_capath_env='SSL_CERT_DIR', openssl_capath='C:\\Program Files\\Common Files\\SSL/certs')
Certifi version: 2026.10.23
Certifi certificate file: d:\AI_ML_Coursework\.venv\Lib\site-packages\certifi\cacert.pem


In [4]:
import os
import certifi

# Configure Python to use Certifi's trusted certificates
os.environ["SSL_CERT_FILE"] = certifi.where()
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()

print("SSL certificate path configured.")
print(certifi.where())

SSL certificate path configured.
d:\AI_ML_Coursework\.venv\Lib\site-packages\certifi\cacert.pem


In [5]:
import urllib.request
import ssl

url = "https://data.brainchip.com/dataset-mirror/cifar10/cifar-10-python.tar.gz"

try:
    context = ssl.create_default_context()
    request = urllib.request.Request(url, method="HEAD")

    with urllib.request.urlopen(request, context=context, timeout=30) as response:
        print("HTTPS connection successful!")
        print("Status code:", response.status)

except Exception as error:
    print("HTTPS connection failed.")
    print(type(error).__name__)
    print(error)

HTTPS connection successful!
Status code: 200


In [6]:
from pathlib import Path
from torchvision import datasets

# Use the alternative CIFAR-10 download mirror
datasets.CIFAR10.url = (
    "https://data.brainchip.com/dataset-mirror/cifar10/"
    "cifar-10-python.tar.gz"
)
# Create the dataset folder
data_dir = Path("../data")
data_dir.mkdir(parents=True, exist_ok=True)

# Load CIFAR-10 training dataset
train_dataset = datasets.CIFAR10(
    root=data_dir,
    train=True,
    download=True
)
# Load CIFAR-10 test dataset
test_dataset = datasets.CIFAR10(
    root=data_dir,
    train=False,
    download=True
)
# Display dataset information
print("Training images:", len(train_dataset))
print("Test images:", len(test_dataset))
print("Number of classes:", len(train_dataset.classes))
print("Class names:", train_dataset.classes)
print("Image shape:", train_dataset.data[0].shape)

100.0%


Training images: 50000
Test images: 10000
Number of classes: 10
Class names: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
Image shape: (32, 32, 3)


In [8]:
#Date Preprocessing and Train-Validation Split
import torch
from torch.utils.data import random_split, DataLoader
from torchvision import transforms

#Define image preprocessing
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        (0.4914, 0.4822, 0.4465),
        (0.2470, 0.2435, 0.2616)
    )
])
#Apply preprocessing to the datasets
train_dataset.transform = transform
test_dataset.transform = transform

#Split training data into training and validation sets
generator = torch.Generator().manual_seed(42)
train_subset, val_subset = random_split(
    train_dataset,
    [45000, 5000],
    generator=generator
)
#Save split indices for reuse in other exercises
torch.save(
    {
        "train_indices": train_subset.indices,
        "validation_indices": val_subset.indices
    },
    data_dir / "cifar10_split_indices.pt"
)
#Create DataLoaders
train_loader = DataLoader(
    train_subset,
    batch_size=128,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_subset,
    batch_size=128,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=0
)
#Display dataset split information
print("Training samples:", len(train_subset))
print("Validation samples:", len(val_subset))
print("Test samples:", len(test_dataset))

#Check one training batch
images, labels = next(iter(train_loader))
print("Batch image shape:", images.shape)
print("Batch label shape:", labels.shape)

Training samples: 45000
Validation samples: 5000
Test samples: 10000
Batch image shape: torch.Size([128, 3, 32, 32])
Batch label shape: torch.Size([128])
